In [ ]:
import math
import torch
import torch.nn as nn

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용할 device:", device)


embedding_dim = 64

query_projection = nn.Linear(embedding_dim, embedding_dim, bias=False)
key_projection = nn.Linear(embedding_dim, embedding_dim, bias=False)
value_projection = nn.Linear(embedding_dim, embedding_dim, bias=False)
output_projection = nn.Linear(embedding_dim, embedding_dim, bias=False)

def multi_head_attention(input_tensor):
    num_heads = 4
    head_dim = embedding_dim // num_heads
    sequence_length = input_tensor.shape[0]

    # Apply Linear Projection
    query = query_projection(input_tensor)
    key = key_projection(input_tensor)
    value = value_projection(input_tensor)

    # multi-head split
    query = query.reshape(sequence_length, num_heads, head_dim).transpose(0, 1)
    key = key.reshape(sequence_length, num_heads, head_dim).transpose(0, 1)
    value = value.reshape(sequence_length, num_heads, head_dim).transpose(0, 1)

    # Attention score (유사도 계산) 
    attention_scores = query@key.transpose(-2, -1) # query 는 현재 [head, token, channel], key 는 [head, channel, token]
    
    # Scaling
    d_k = key.shape[-1]
    scaled_scores = attention_scores/math.sqrt(d_k)

    # Attention_weight (가중치화 (확률로 바꾸기))
    attention_weights = torch.softmax(scaled_scores, dim=-1)

    # Attention 결과
    attention_output = attention_weights @ value

    # Output Projection
    attention_output = attention_output.transpose(0, 1).reshape(sequence_length, embedding_dim)
    attention_output = output_projection(attention_output)

    return attention_output

---
# Part 2. LayerNorm, Feed Forward, Residual — 인코더의 나머지 세 부품

## 2.1 LayerNorm (정규화)

부품을 수십 층 쌓으면 중간 값들이 점점 커지거나 작아져 **학습이 불안정**해집니다. <br>
LayerNorm은 **각 토큰 벡터를 "평균 0, 표준편차 1"로 리셋**해서 값의 스케일을 일정하게 유지해 줍니다. <br>
인코더 레이어마다 2번씩 들어가는 필수 부품입니다.

In [ ]:
torch.manual_seed(42)

input_tensor = torch.rand(6, 64)

# 각 토큰의 평균과 분산 계산
token_mean = 
token_variance = 

# 정규화: (x - 평균) / sqrt(분산 + epsilon)
epsilon = 1e-5


# 함수로 정리
def layer_norm(input_tensor, epsilon=1e-5):
    """함수로 정리해보세요"""
    
    return

print("\nlayer_norm 함수 출력:", layer_norm(input_tensor).shape)   # 예상: (6, 64)

In [ ]:
# Learnable Parameter Version (정규화된 표현을 모델이 원하는 분포로 조금 더 조정할 수 있게)
def layer_norm(input_tensor, epsilon=1e-5):
    """함수로 정리해보세요"""
    
    return 

print("\nlayer_norm 함수 출력:", layer_norm(input_tensor).shape)   # 예상: (6, 64)

## 2.2 Residual Connection — 입력을 보존하는 지름길

```python
output = input_tensor + layer(input_tensor)
```

In [ ]:
# Residual Connection



## 2.3 Feed Forward Network (FFN) — "채널 간" 정보 섞기

**Attention 은 토큰간 정보를, FFN 은 토큰 내 channel 간 정보를 섞는다**

| | Attention | Feed Forward (FFN) |
|---|---|---|
| 무엇을 섞나 | **토큰 간** (cat이 mat을 쳐다봄) | **채널 간** (한 토큰 안의 64개 숫자끼리) |
| 섞이는 차원 | `seq_len` 차원 (행 방향) | `embedding_dim` 차원 (열 방향) |
| 토큰끼리 영향? | ✅ 서로 영향을 줌 | ❌ 각 토큰이 **독립적으로** 처리됨 |

In [ ]:
torch.manual_seed(42)

# 두 개의 Linear: "확장"과 "축소"


# shape 관찰

print("확장 후:", expanded.shape)                  # 예상: (6, 256)


print("ReLU 후:", activated.shape)                # 예상: (6, 256)


print("축소 후:", ffn_result.shape)                # 예상: (6, 64)


# 이제 함수로 정리!
def feed_forward(input_tensor):
    return
